# Customer Churn Prediction — Data Preprocessing

## 1. Introduction

This notebook builds a **clean, reusable, leakage-safe preprocessing workflow** on top of the
findings from `01_data_exploration.ipynb`. It does **not** train any machine learning model —
that happens in `03_model_training_evaluation.ipynb`.

The goal here is to produce:
- A cleaned version of the dataset (fixed `TotalCharges`, no `customerID`, encoded `Churn`)
- A stratified train/test split, performed **before** any data-dependent preprocessing
- An automatically-detected list of numerical vs categorical features
- A reusable `ColumnTransformer` (imputation + scaling + one-hot encoding), fit **only** on
  training data
- A pipeline architecture (`preprocessing → SMOTE → classifier`) that ensures SMOTE only ever
  touches training data
- A set of explicit validation checks confirming none of the above leaks information from the
  test set into training

All of the actual logic lives in `src/data_preprocessing.py`, which this notebook imports and
demonstrates — that way the exact same, tested functions can be reused later in `src/train.py`
and in the Streamlit app, instead of being copy-pasted and risking drift between them.


## 2. Import Libraries

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make src/ importable regardless of where Jupyter was launched from
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import data_preprocessing as dp

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)


Project root: /path/to/customer-churn-prediction
Raw data path: /path/to/customer-churn-prediction/data/raw/Telco-Customer-Churn.csv


## 3. Load Raw Dataset

We use `dp.load_data()`, which simply reads the CSV with no modification. We will keep a
reference to this raw dataframe throughout the notebook purely to *verify* it is never altered.

In [2]:
raw_df = dp.load_data(RAW_DATA_PATH)
raw_df_fingerprint_before = pd.util.hash_pandas_object(raw_df).sum()  # used later to prove immutability

print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()


Raw dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 4. Data Cleaning

### 4.1 Investigate `TotalCharges`

Before cleaning, let's re-confirm (independently of Notebook 1) exactly which rows have a blank
`TotalCharges`, and what those rows have in common.

In [3]:
blank_mask = raw_df["TotalCharges"].astype(str).str.strip() == ""
blank_rows = raw_df[blank_mask]
print(f"Number of rows with blank TotalCharges: {len(blank_rows)}")
blank_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]


Number of rows with blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [4]:
print("Unique tenure values among the blank-TotalCharges rows:", blank_rows["tenure"].unique())
print("Unique Churn values among the blank-TotalCharges rows:", blank_rows["Churn"].unique())


Unique tenure values among the blank-TotalCharges rows: [0]
Unique Churn values among the blank-TotalCharges rows: <StringArray>
['No']
Length: 1, dtype: str


**Finding:** every single row with a blank `TotalCharges` has `tenure == 0`. These are
customers who joined but have not yet completed a billing cycle, so they have no accumulated
charges yet — not truly "missing" in the sense of being lost or corrupted data, but a real zero
that was recorded as blank text.

### 4.2 Decision on how to handle these values

We fill these with **0**, for two reasons:
1. **Domain logic:** `TotalCharges` should conceptually equal `MonthlyCharges × tenure`. With
   `tenure == 0`, that product is exactly `0` — so `0` is not a guess, it is the logically correct
   value.
2. **No leakage risk:** because this rule is a fixed, deterministic transformation (it does not
   depend on any statistic — like a mean or median — computed from the data), it is safe to apply
   identically to the whole dataset *before* the train/test split, unlike e.g. mean/median
   imputation, which would have to be learned from training data only.

### 4.3 Run the full cleaning function

`dp.clean_data()` performs, in order: `TotalCharges` text→numeric conversion and 0-filling,
dropping `customerID`, and encoding `Churn` as 0/1.

In [5]:
df_clean = dp.clean_data(raw_df)

print(f"Cleaned shape: {df_clean.shape}  (one fewer column than raw: customerID was dropped)")
print(f"TotalCharges dtype after cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Missing values remaining in TotalCharges: {df_clean['TotalCharges'].isnull().sum()}")
print(f"Churn dtype after cleaning: {df_clean['Churn'].dtype}, unique values: {sorted(df_clean['Churn'].unique())}")
print(f"'customerID' present in cleaned columns: {'customerID' in df_clean.columns}")
df_clean.head()


Cleaned shape: (7043, 20)  (one fewer column than raw: customerID was dropped)
TotalCharges dtype after cleaning: float64
Missing values remaining in TotalCharges: 0
Churn dtype after cleaning: int64, unique values: [np.int64(0), np.int64(1)]
'customerID' present in cleaned columns: False


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [6]:
# Confirm the raw CSV on disk, and the in-memory raw_df, were NOT modified by cleaning
raw_df_fingerprint_after = pd.util.hash_pandas_object(raw_df).sum()
assert raw_df_fingerprint_before == raw_df_fingerprint_after, "raw_df was unexpectedly modified!"
print("Confirmed: raw_df is unchanged after clean_data() (clean_data returns a separate copy).")

raw_df_reloaded = pd.read_csv(RAW_DATA_PATH)
assert raw_df_reloaded["TotalCharges"].dtype == object or raw_df_reloaded["TotalCharges"].dtype.name in ("object", "str"), \
    "Raw CSV on disk appears to have been modified (TotalCharges is no longer text)."
print("Confirmed: the raw CSV file on disk is unchanged (TotalCharges is still stored as text there).")


Confirmed: raw_df is unchanged after clean_data() (clean_data returns a separate copy).
Confirmed: the raw CSV file on disk is unchanged (TotalCharges is still stored as text there).


## 5. Target and Feature Separation

We split the cleaned dataframe into a feature matrix `X` and target vector `y` using
`dp.split_features_target()`.

In [7]:
X, y = dp.split_features_target(df_clean)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"'customerID' in X.columns: {'customerID' in X.columns}")
print(f"'Churn' in X.columns: {'Churn' in X.columns}")
print()
print("Feature names in X:")
for i, col in enumerate(X.columns, start=1):
    print(f"{i:>2}. {col}")


X shape: (7043, 19)
y shape: (7043,)
'customerID' in X.columns: False
'Churn' in X.columns: False

Feature names in X:
 1. gender
 2. SeniorCitizen
 3. Partner
 4. Dependents
 5. tenure
 6. PhoneService
 7. MultipleLines
 8. InternetService
 9. OnlineSecurity
10. OnlineBackup
11. DeviceProtection
12. TechSupport
13. StreamingTV
14. StreamingMovies
15. Contract
16. PaperlessBilling
17. PaymentMethod
18. MonthlyCharges
19. TotalCharges


In [8]:
print("Target distribution (y):")
print(y.value_counts())
print()
print("Target distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))


Target distribution (y):
Churn
0    5174
1    1869
Name: count, dtype: int64

Target distribution (%):
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


## 6. Train/Test Split

We use `dp.create_train_test_split()`, which performs an **80/20 stratified split**
(`stratify=y`) with `random_state=42` for reproducibility.

**Why must this happen before any data-dependent preprocessing?**
Any transformer that *learns* something from the data (a `StandardScaler`'s mean/std, a
`SimpleImputer`'s median, which categories a `OneHotEncoder` has seen) must only learn from
training data. If we fit such a transformer on the full dataset first and split afterward, the
test set's information (its own distribution, its own values) would have quietly influenced the
transformation applied to it — meaning the "unseen" test set is no longer truly unseen. That
inflates evaluation metrics and gives a false sense of how the model will perform on genuinely new
customers. Splitting first, and fitting only on the training partition, is the only way to get an
honest estimate of generalization performance later.

In [9]:
X_train, X_test, y_train, y_test = dp.create_train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


X_train shape: (5634, 19)
X_test shape:  (1409, 19)
y_train shape: (5634,)
y_test shape:  (1409,)


In [10]:
print("Target distribution in y_train:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))
print()
print("Target distribution in y_test:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))
print()
print(f"Overall churn rate: {round(y.mean() * 100, 2)}% | "
      f"Train churn rate: {round(y_train.mean() * 100, 2)}% | "
      f"Test churn rate: {round(y_test.mean() * 100, 2)}%")
print("Stratification confirmed: train and test churn rates closely match the overall rate.")


Target distribution in y_train:
Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Target distribution in y_test:
Churn
0    1035
1     374
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Overall churn rate: 26.54% | Train churn rate: 26.54% | Test churn rate: 26.54%
Stratification confirmed: train and test churn rates closely match the overall rate.


## 7. Feature Type Identification

`dp.get_feature_types()` automatically detects numerical vs categorical columns from `X`'s dtypes
— nothing is hardcoded. Numeric dtypes (which include `SeniorCitizen`, since it's stored as 0/1
integers) become numerical features; text/object columns become categorical features.

In [11]:
numerical_features, categorical_features = dp.get_feature_types(X_train)

print(f"Numerical features ({len(numerical_features)}):")
for f in numerical_features:
    print(f"  - {f}")

print(f"\nCategorical features ({len(categorical_features)}):")
for f in categorical_features:
    print(f"  - {f}")

print(f"\nTotal features accounted for: {len(numerical_features) + len(categorical_features)} "
      f"(X has {X_train.shape[1]} columns)")


Numerical features (4):
  - SeniorCitizen
  - tenure
  - MonthlyCharges
  - TotalCharges

Categorical features (15):
  - gender
  - Partner
  - Dependents
  - PhoneService
  - MultipleLines
  - InternetService
  - OnlineSecurity
  - OnlineBackup
  - DeviceProtection
  - TechSupport
  - StreamingTV
  - StreamingMovies
  - Contract
  - PaperlessBilling
  - PaymentMethod

Total features accounted for: 19 (X has 19 columns)


## 8. Preprocessing Pipeline

`dp.build_preprocessor()` returns an **unfitted** `ColumnTransformer`:
- Numerical features → median imputation (safety net) → `StandardScaler`
- Categorical features → most-frequent imputation (safety net) → `OneHotEncoder(handle_unknown="ignore")`

We fit it **only on `X_train`**, then only ever call `.transform()` on `X_test`.

In [12]:
preprocessor = dp.build_preprocessor(numerical_features, categorical_features)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [13]:
preprocessor.fit(X_train)  # fit ONLY on training data

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train_transformed shape: {X_train_transformed.shape}")
print(f"X_test_transformed shape:  {X_test_transformed.shape}")
print("(Test set was transformed using parameters learned from the training set only.)")


X_train_transformed shape: (5634, 45)
X_test_transformed shape:  (1409, 45)
(Test set was transformed using parameters learned from the training set only.)


In [14]:
# Concrete proof that the scaler learned its statistics from TRAIN only, not the full dataset
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
tenure_idx = numerical_features.index("tenure")

fitted_mean = scaler.mean_[tenure_idx]
actual_train_mean = X_train["tenure"].mean()
actual_full_mean = X["tenure"].mean()

print(f"Scaler's fitted mean for 'tenure':      {fitted_mean:.4f}")
print(f"Actual TRAIN-only mean for 'tenure':    {actual_train_mean:.4f}")
print(f"Actual FULL-DATASET mean for 'tenure':  {actual_full_mean:.4f}")

assert np.isclose(fitted_mean, actual_train_mean), "Leakage suspected: scaler mean does not match train-only mean!"
print("\nConfirmed: the scaler's learned mean matches the TRAIN-only mean, not the full-dataset mean -> no leakage.")


Scaler's fitted mean for 'tenure':      32.4851
Actual TRAIN-only mean for 'tenure':    32.4851
Actual FULL-DATASET mean for 'tenure':  32.3711

Confirmed: the scaler's learned mean matches the TRAIN-only mean, not the full-dataset mean -> no leakage.


## 9. SMOTE Pipeline Design

**Why SMOTE must only ever be applied to training data, inside the pipeline:**
SMOTE (Synthetic Minority Over-sampling Technique) creates *synthetic* new examples of the
minority class (`Churn = Yes`) by interpolating between existing minority examples. If SMOTE were
applied to the full dataset before splitting, synthetic points derived from what becomes "test"
data could end up influencing training, and — worse — synthetic points could end up in the test
set itself, meaning the model would be evaluated on fake, non-real customers. Both problems
inflate reported performance and make it meaningless as an estimate of real-world accuracy. SMOTE
must therefore run strictly *after* the split and *only* on the training partition.

`dp.build_pipeline()` assembles: **preprocessing → SMOTE → (optional) classifier**, using
`imblearn.pipeline.Pipeline`. This specific pipeline class (not plain scikit-learn's `Pipeline`)
understands that a resampling step should only run during training-time `.fit()`/`.fit_resample()`
calls, and is automatically skipped when the pipeline is used to `.transform()` or `.predict()` on
new data — so it is architecturally impossible for this pipeline to apply SMOTE to test data or to
a new customer in the Streamlit app.

**No classifier is trained in this step.** Below we build the pipeline with `classifier=None` to
demonstrate the architecture, and only call `.fit_resample()` (not `.fit()` with a model) purely to
verify that the resampling behaves as expected on the training data.

In [15]:
smote_pipeline = dp.build_pipeline(
    preprocessor=dp.build_preprocessor(numerical_features, categorical_features),
    classifier=None,       # no model yet - this step is about preprocessing/SMOTE architecture only
    use_smote=True,
    random_state=42,
)

print("Pipeline steps:")
for name, step in smote_pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")


Pipeline steps:
  - preprocessor: ColumnTransformer
  - smote: SMOTE


In [16]:
# Demonstrate the resampling effect on the TRAINING data only (no model fit here)
X_train_resampled, y_train_resampled = smote_pipeline.fit_resample(X_train, y_train)

print("Training class distribution BEFORE SMOTE:")
print(y_train.value_counts())
print()
print("Training class distribution AFTER SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print()
print(f"Test set size (must be completely unaffected): {X_test.shape[0]} rows (unchanged)")


Training class distribution BEFORE SMOTE:
Churn
0    4139
1    1495
Name: count, dtype: int64

Training class distribution AFTER SMOTE:
Churn
0    4139
1    4139
Name: count, dtype: int64

Test set size (must be completely unaffected): 1409 rows (unchanged)


Note the shape of `X_train_resampled` above: it is the *transformed* (numeric, one-hot
encoded) feature matrix, already balanced. This exact same `smote_pipeline` structure — with a
real classifier appended as the final step — is what `src/train.py` will use in Step 4 to fit
Logistic Regression, Random Forest, and XGBoost, each fully protected from data leakage.

## 10. Leakage Checks

We now run a battery of explicit checks confirming the pipeline is leakage-safe.

In [17]:
checks_passed = []

# 1. Test set size never changed by any step above
checks_passed.append(("Test set row count unchanged", X_test.shape[0] == y_test.shape[0] == int(round(len(X) * 0.20))))

# 2. Scaler statistics come from train only (re-verified explicitly)
checks_passed.append(("Scaler mean matches train-only mean", np.isclose(
    preprocessor.named_transformers_["num"].named_steps["scaler"].mean_[numerical_features.index("tenure")],
    X_train["tenure"].mean()
)))

# 3. customerID is not among the features used for modeling
checks_passed.append(("customerID excluded from X", "customerID" not in X.columns))

# 4. Preprocessor was fit exactly once, and only on X_train (5634 rows), not X (7043) or X_test (1409)
checks_passed.append(("Preprocessor n_features_in_ matches X_train columns", preprocessor.n_features_in_ == X_train.shape[1]))

# 5. SMOTE only present in a pipeline that is invoked with fit/fit_resample on TRAIN data in this notebook
checks_passed.append(("SMOTE step present in pipeline", "smote" in dict(smote_pipeline.steps)))

for description, passed in checks_passed:
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {description}")

assert all(passed for _, passed in checks_passed), "One or more leakage checks failed!"
print("\nAll leakage checks passed.")


[PASS] Test set row count unchanged
[PASS] Scaler mean matches train-only mean
[PASS] customerID excluded from X
[PASS] Preprocessor n_features_in_ matches X_train columns
[PASS] SMOTE step present in pipeline

All leakage checks passed.


## 11. Validation

We now run the full data-quality / structural validation suite from `src/data_preprocessing.py`
(`run_all_validations`), plus a couple of additional checks specific to this notebook.

In [18]:
# Full validation suite (raises AssertionError on failure)
dp.run_all_validations(raw_df, X, y, numerical_features, categorical_features)
print("run_all_validations(): all checks passed.")


run_all_validations(): all checks passed.


In [19]:
# Additional check: are there any categories in the test set unseen during training?
unseen_categories = dp.validate_categories_known(X_train, X_test, categorical_features)

if unseen_categories:
    print("Categories present in test but not in train (handled safely by handle_unknown='ignore'):")
    for col, cats in unseen_categories.items():
        print(f"  - {col}: {cats}")
else:
    print("No unseen categories found in the test set - every category in test also appears in train.")


No unseen categories found in the test set - every category in test also appears in train.


In [20]:
# Explicit re-confirmation of individual requirements from the task spec
assert "customerID" not in X.columns, "customerID leaked into features!"
assert set(y.unique()) == {0, 1}, "Churn not correctly encoded as 0/1!"
assert abs(y_train.mean() - y_test.mean()) < 0.02, "Stratification looks off between train and test!"
assert raw_df["TotalCharges"].dtype == object or raw_df["TotalCharges"].dtype.name in ("object", "str"), \
    "Raw dataframe TotalCharges dtype changed unexpectedly (should still be text in the raw df)."

print("customerID excluded from X: confirmed")
print("Churn correctly encoded as 0/1: confirmed")
print(f"Stratification: train churn rate {y_train.mean():.4f} vs test churn rate {y_test.mean():.4f} - close match, confirmed")
print("Raw dataframe/CSV unaffected by cleaning: confirmed")


customerID excluded from X: confirmed
Churn correctly encoded as 0/1: confirmed
Stratification: train churn rate 0.2654 vs test churn rate 0.2654 - close match, confirmed
Raw dataframe/CSV unaffected by cleaning: confirmed


## 12. Summary and Next Steps

**What this notebook established (see the executed cell outputs above for exact values):**
- `TotalCharges` was converted to numeric and its 11 blank values (all `tenure == 0` customers)
  were filled with `0`, based on domain logic rather than a statistic learned from data.
- `customerID` was dropped from the feature set; `Churn` was encoded as 0/1.
- Features and target were separated into `X` and `y`.
- A stratified 80/20 train/test split was performed with `random_state=42`, **before** any
  data-dependent preprocessing.
- Numerical and categorical feature lists were derived automatically from dtypes (no hardcoding).
- A `ColumnTransformer` (imputation + scaling + one-hot encoding) was built and fit **only** on
  the training set; the scaler's learned mean was explicitly verified to match the train-only
  mean, not the full-dataset mean.
- A `preprocessing → SMOTE` pipeline (via `imblearn.pipeline.Pipeline`) was constructed and
  demonstrated on the training data only, with the test set left completely untouched. No
  classifier was trained.
- A full suite of leakage and data-quality validation checks was run and passed.

**Note on `data/processed/`:** per the project's anti-leakage rules, we deliberately do **not**
save a globally scaled/encoded dataset there — doing so and later loading it for modeling would
silently reintroduce leakage (the encoder/scaler would have "seen" the whole dataset, test
included). The only artifact that should ever be persisted from preprocessing is the **fitted
pipeline object itself**, trained strictly on the training split — and that will be saved as part
of `models/churn_model.pkl` once training happens in Step 4, not here.

**What Step 4 (`src/train.py` + `03_model_training_evaluation.ipynb`) should do next:**
1. Reuse `dp.load_data`, `dp.clean_data`, `dp.split_features_target`, `dp.create_train_test_split`,
   `dp.get_feature_types`, `dp.build_preprocessor`, and `dp.build_pipeline` exactly as demonstrated
   here.
2. Attach a real classifier (Logistic Regression, Random Forest, XGBoost in turn) as the final
   pipeline step.
3. Fit each full pipeline (preprocessing → SMOTE → classifier) on `X_train`/`y_train` only.
4. Evaluate each fitted pipeline on the untouched `X_test`/`y_test` using Accuracy, Precision,
   Recall, F1, ROC-AUC, and a Confusion Matrix.
5. Compare the three models honestly and select the best one based on the actual metrics.
6. Run SHAP on the selected model to explain which features drive its predictions.
7. Save the winning fitted pipeline with `joblib` to `models/churn_model.pkl`.
